# 10 — Vulnerability Assessment & CVE Analysis

**Author:** Bency (Benjamin Adjei)  
**Course:** M.S. Cybersecurity — Risk Management & Vulnerability Analysis  
**Tools:** Python 3, requests, json, NIST NVD API

---

## Objectives
- Understand the vulnerability management lifecycle
- Query the NIST National Vulnerability Database (NVD) API for CVE data
- Parse CVSS scores and severity ratings
- Map discovered services to known CVEs
- Prioritize vulnerabilities by risk score
- Generate a remediation-ready vulnerability report

> 📌 **NVD API:** Free, no API key required for basic queries. Rate limit: 5 requests/30s (unauthenticated).

## 1. Setup & Imports

In [ ]:
import json
import time
from datetime import datetime

# requests is needed for live NVD API calls
# !pip install requests
try:
    import requests
    REQUESTS_AVAILABLE = True
except ImportError:
    REQUESTS_AVAILABLE = False
    print('requests not installed — using simulated CVE data for demo')

NVD_API_BASE = 'https://services.nvd.nist.gov/rest/json/cves/2.0'

## 2. Vulnerability Management Lifecycle

```
  Discover  →  Assess  →  Prioritize  →  Remediate  →  Verify  →  Report
     ↑                                                                |
     └────────────────────── Continuous Loop ───────────────────────┘
```

### CVSS Scoring (Common Vulnerability Scoring System)

| CVSS Score | Severity | Action Timeline |
|------------|----------|-----------------|
| 9.0 – 10.0 | **Critical** | Patch within 24 hours |
| 7.0 – 8.9  | **High**     | Patch within 7 days |
| 4.0 – 6.9  | **Medium**   | Patch within 30 days |
| 0.1 – 3.9  | **Low**      | Patch within 90 days |
| 0.0        | **None**     | Informational only |

### Key CVE Fields
- **CVE ID** — Unique identifier (e.g. CVE-2024-1234)
- **CVSS v3 Score** — Numeric risk rating 0–10
- **Vector String** — Encodes attack complexity, privileges needed, impact
- **CWE** — Weakness category (e.g. CWE-79 = XSS, CWE-89 = SQL Injection)
- **CPE** — Affected software/hardware platform

## 3. Simulated CVE Database

Based on real CVEs for common services discovered during recon.

In [ ]:
# Simulated CVE data (mirrors NVD API response structure)
SIMULATED_CVES = [
    {
        'id'         : 'CVE-2023-0386',
        'description': 'Linux kernel OverlayFS privilege escalation allows local attacker to gain root.',
        'cvss_score' : 7.8,
        'severity'   : 'HIGH',
        'vector'     : 'CVSS:3.1/AV:L/AC:L/PR:L/UI:N/S:U/C:H/I:H/A:H',
        'cwe'        : 'CWE-282',
        'service'    : 'Linux Kernel < 6.2',
        'published'  : '2023-03-22',
        'patch'      : 'Upgrade kernel to 6.2+'
    },
    {
        'id'         : 'CVE-2023-23397',
        'description': 'Microsoft Outlook NTLM hash leak via calendar invite. No user interaction required.',
        'cvss_score' : 9.8,
        'severity'   : 'CRITICAL',
        'vector'     : 'CVSS:3.1/AV:N/AC:L/PR:N/UI:N/S:U/C:H/I:H/A:H',
        'cwe'        : 'CWE-294',
        'service'    : 'Microsoft Outlook',
        'published'  : '2023-03-14',
        'patch'      : 'Apply Microsoft March 2023 Patch Tuesday update'
    },
    {
        'id'         : 'CVE-2021-44228',
        'description': 'Log4Shell: Remote code execution in Apache Log4j 2 via JNDI injection in log messages.',
        'cvss_score' : 10.0,
        'severity'   : 'CRITICAL',
        'vector'     : 'CVSS:3.1/AV:N/AC:L/PR:N/UI:N/S:C/C:H/I:H/A:H',
        'cwe'        : 'CWE-917',
        'service'    : 'Apache Log4j 2.x < 2.15.0',
        'published'  : '2021-12-10',
        'patch'      : 'Upgrade Log4j to 2.17.1+; set log4j2.formatMsgNoLookups=true'
    },
    {
        'id'         : 'CVE-2022-0847',
        'description': 'Dirty Pipe: Linux kernel pipe privilege escalation allows overwriting read-only files.',
        'cvss_score' : 7.8,
        'severity'   : 'HIGH',
        'vector'     : 'CVSS:3.1/AV:L/AC:L/PR:L/UI:N/S:U/C:H/I:H/A:H',
        'cwe'        : 'CWE-269',
        'service'    : 'Linux Kernel 5.8 – 5.16.10',
        'published'  : '2022-03-07',
        'patch'      : 'Upgrade kernel to 5.16.11, 5.15.25, or 5.10.102+'
    },
    {
        'id'         : 'CVE-2023-44487',
        'description': 'HTTP/2 Rapid Reset Attack: allows unauthenticated DoS against web servers.',
        'cvss_score' : 7.5,
        'severity'   : 'HIGH',
        'vector'     : 'CVSS:3.1/AV:N/AC:L/PR:N/UI:N/S:U/C:N/I:N/A:H',
        'cwe'        : 'CWE-400',
        'service'    : 'HTTP/2 web servers (nginx, Apache, IIS)',
        'published'  : '2023-10-10',
        'patch'      : 'Apply vendor patches; rate-limit HTTP/2 RESET frames'
    },
    {
        'id'         : 'CVE-2023-20198',
        'description': 'Cisco IOS XE Web UI privilege escalation — unauthenticated remote admin account creation.',
        'cvss_score' : 10.0,
        'severity'   : 'CRITICAL',
        'vector'     : 'CVSS:3.1/AV:N/AC:L/PR:N/UI:N/S:C/C:H/I:H/A:H',
        'cwe'        : 'CWE-306',
        'service'    : 'Cisco IOS XE Web UI',
        'published'  : '2023-10-16',
        'patch'      : 'Disable HTTP server; apply Cisco advisory patch'
    },
    {
        'id'         : 'CVE-2024-3400',
        'description': 'PAN-OS command injection in GlobalProtect Gateway — unauthenticated RCE as root.',
        'cvss_score' : 10.0,
        'severity'   : 'CRITICAL',
        'vector'     : 'CVSS:3.1/AV:N/AC:L/PR:N/UI:N/S:C/C:H/I:H/A:H',
        'cwe'        : 'CWE-77',
        'service'    : 'Palo Alto PAN-OS 10.2 / 11.0 / 11.1',
        'published'  : '2024-04-12',
        'patch'      : 'Apply PAN-OS hotfix; disable GlobalProtect if not in use'
    },
    {
        'id'         : 'CVE-2023-34362',
        'description': 'MOVEit Transfer SQL injection allows unauthenticated access to database content.',
        'cvss_score' : 9.8,
        'severity'   : 'CRITICAL',
        'vector'     : 'CVSS:3.1/AV:N/AC:L/PR:N/UI:N/S:U/C:H/I:H/A:H',
        'cwe'        : 'CWE-89',
        'service'    : 'Progress MOVEit Transfer',
        'published'  : '2023-06-02',
        'patch'      : 'Apply May 2023 MOVEit security patch immediately'
    }
]

print(f'Loaded {len(SIMULATED_CVES)} CVEs into local database')

## 4. Query the NIST NVD API (Live)

In [ ]:
def query_nvd(keyword: str, results_per_page: int = 5) -> list:
    """Query the NIST NVD API for CVEs matching a keyword."""
    if not REQUESTS_AVAILABLE:
        print('requests not available — returning simulated results')
        return [c for c in SIMULATED_CVES if keyword.lower() in c['description'].lower()]

    params = {'keywordSearch': keyword, 'resultsPerPage': results_per_page}
    headers = {'User-Agent': 'cybersecurity-lab-notebook/1.0'}

    try:
        response = requests.get(NVD_API_BASE, params=params, headers=headers, timeout=10)
        response.raise_for_status()
        data = response.json()

        cves = []
        for item in data.get('vulnerabilities', []):
            cve  = item['cve']
            desc = next((d['value'] for d in cve['descriptions'] if d['lang'] == 'en'), 'N/A')

            # Extract CVSS v3 score
            score    = None
            severity = 'UNKNOWN'
            metrics  = cve.get('metrics', {})
            for key in ['cvssMetricV31', 'cvssMetricV30']:
                if key in metrics:
                    cvss_data = metrics[key][0]['cvssData']
                    score     = cvss_data.get('baseScore')
                    severity  = cvss_data.get('baseSeverity', 'UNKNOWN')
                    break

            cves.append({
                'id'         : cve['id'],
                'description': desc[:200],
                'cvss_score' : score,
                'severity'   : severity,
                'published'  : cve.get('published', 'N/A')[:10]
            })
        return cves

    except requests.RequestException as e:
        print(f'NVD API error: {e} — falling back to simulated data')
        return [c for c in SIMULATED_CVES if keyword.lower() in c['description'].lower()]


# Live query example — search for Log4j CVEs
print('Querying NVD for Log4j CVEs...\n')
log4j_cves = query_nvd('Log4j', results_per_page=3)
for cve in log4j_cves:
    print(f"  {cve['id']}  CVSS={cve['cvss_score']}  [{cve['severity']}]")
    print(f"    {cve['description'][:120]}...\n")

time.sleep(1)  # Respect NVD rate limit

## 5. Vulnerability Prioritization

In [ ]:
def get_severity_label(score: float) -> str:
    if score >= 9.0: return 'CRITICAL'
    if score >= 7.0: return 'HIGH'
    if score >= 4.0: return 'MEDIUM'
    if score > 0.0:  return 'LOW'
    return 'NONE'


def get_patch_sla(severity: str) -> str:
    return {
        'CRITICAL': 'Patch within 24 hours',
        'HIGH'    : 'Patch within 7 days',
        'MEDIUM'  : 'Patch within 30 days',
        'LOW'     : 'Patch within 90 days'
    }.get(severity, 'Review as needed')


# Sort all CVEs by score descending
prioritized = sorted(SIMULATED_CVES, key=lambda x: x['cvss_score'], reverse=True)

print('=== VULNERABILITY PRIORITY LIST ===\n')
print(f'{"CVE ID":<20} {"Score":<7} {"Severity":<10} {"SLA":<30} {"Service"}')
print('-' * 100)
for v in prioritized:
    sla = get_patch_sla(v['severity'])
    print(f"{v['id']:<20} {v['cvss_score']:<7} {v['severity']:<10} {sla:<30} {v['service']}")

## 6. CVSS Vector String Analysis

In [ ]:
CVSS_LABELS = {
    'AV' : {'N': 'Network', 'A': 'Adjacent', 'L': 'Local', 'P': 'Physical'},
    'AC' : {'L': 'Low', 'H': 'High'},
    'PR' : {'N': 'None', 'L': 'Low', 'H': 'High'},
    'UI' : {'N': 'None', 'R': 'Required'},
    'S'  : {'U': 'Unchanged', 'C': 'Changed'},
    'C'  : {'N': 'None', 'L': 'Low', 'H': 'High'},
    'I'  : {'N': 'None', 'L': 'Low', 'H': 'High'},
    'A'  : {'N': 'None', 'L': 'Low', 'H': 'High'},
}

def parse_cvss_vector(vector: str) -> dict:
    """Parse a CVSS v3 vector string into human-readable components."""
    parts  = vector.split('/')  # e.g. CVSS:3.1/AV:N/AC:L/...
    parsed = {}
    for part in parts[1:]:
        key, val = part.split(':')
        label    = CVSS_LABELS.get(key, {}).get(val, val)
        parsed[key] = {'code': val, 'label': label}
    return parsed


# Analyse Log4Shell vector
log4shell = next(c for c in SIMULATED_CVES if c['id'] == 'CVE-2021-44228')
parsed    = parse_cvss_vector(log4shell['vector'])

print(f"CVSS Vector Analysis: {log4shell['id']} (Score: {log4shell['cvss_score']})\n")
field_names = {
    'AV': 'Attack Vector', 'AC': 'Attack Complexity',
    'PR': 'Privileges Req.', 'UI': 'User Interaction',
    'S' : 'Scope', 'C': 'Confidentiality', 'I': 'Integrity', 'A': 'Availability'
}
for key, info in parsed.items():
    name = field_names.get(key, key)
    print(f"  {name:<22}: {info['label']} ({info['code']})")

## 7. Service-to-CVE Mapping

Map services discovered during recon (Notebook 09) to known vulnerabilities.

In [ ]:
# Simulated services discovered from port scan (from Notebook 09)
DISCOVERED_SERVICES = [
    {'port': 22,   'service': 'OpenSSH 7.4',         'banner': 'SSH-2.0-OpenSSH_7.4'},
    {'port': 80,   'service': 'Apache HTTP 2.4.49',   'banner': 'Apache/2.4.49'},
    {'port': 443,  'service': 'nginx 1.18.0',         'banner': 'nginx/1.18.0'},
    {'port': 3306, 'service': 'MySQL 5.7.38',         'banner': 'MySQL 5.7.38'},
    {'port': 8080, 'service': 'Apache Tomcat 9.0.50', 'banner': 'Apache Tomcat/9.0.50'},
]

# CVE lookup table for discovered service versions
SERVICE_CVE_MAP = {
    'Apache HTTP 2.4.49': [
        {'id': 'CVE-2021-41773', 'score': 9.8, 'severity': 'CRITICAL',
         'desc': 'Path traversal and RCE in Apache HTTP Server 2.4.49',
         'patch': 'Upgrade to Apache 2.4.51+'}
    ],
    'Apache Tomcat 9.0.50': [
        {'id': 'CVE-2022-34305', 'score': 6.1, 'severity': 'MEDIUM',
         'desc': 'XSS in Tomcat examples web application',
         'patch': 'Upgrade to Tomcat 9.0.65+; remove example apps'}
    ],
    'MySQL 5.7.38': [
        {'id': 'CVE-2022-21595', 'score': 4.4, 'severity': 'MEDIUM',
         'desc': 'MySQL Server privilege escalation via C API',
         'patch': 'Upgrade to MySQL 5.7.39+'}
    ],
    'OpenSSH 7.4': [
        {'id': 'CVE-2023-38408', 'score': 9.8, 'severity': 'CRITICAL',
         'desc': 'Remote code execution via ssh-agent forwarding in OpenSSH < 9.3p2',
         'patch': 'Upgrade to OpenSSH 9.3p2+'}
    ]
}

print('=== SERVICE VULNERABILITY MAPPING ===\n')
findings = []
for svc in DISCOVERED_SERVICES:
    cves = SERVICE_CVE_MAP.get(svc['service'], [])
    status = f'{len(cves)} CVE(s) found' if cves else 'No known CVEs matched'
    print(f"  Port {svc['port']:5} | {svc['service']:<25} | {status}")
    for cve in cves:
        findings.append({**cve, 'port': svc['port'], 'service': svc['service']})
        print(f"    [{cve['severity']:8}] {cve['id']}  Score={cve['score']}  {cve['desc'][:70]}")

## 8. Vulnerability Assessment Report

In [ ]:
from collections import Counter

all_findings = findings + [
    {k: v for k, v in c.items() if k != 'vector'} for c in SIMULATED_CVES
]

va_report = {
    'report_generated'  : datetime.now().strftime('%Y-%m-%d %H:%M:%S'),
    'assessment_scope'  : 'Internal lab environment — localhost simulation',
    'total_services'    : len(DISCOVERED_SERVICES),
    'total_cves_matched': len(findings),
    'severity_breakdown': dict(Counter(f['severity'] for f in findings)),
    'critical_findings' : [
        f for f in findings if f['severity'] == 'CRITICAL'
    ],
    'remediation_plan'  : [
        {
            'priority': i + 1,
            'cve'     : f['id'],
            'score'   : f['score'],
            'service' : f['service'],
            'action'  : f['patch'],
            'sla'     : get_patch_sla(f['severity'])
        }
        for i, f in enumerate(
            sorted(findings, key=lambda x: x['score'], reverse=True)
        )
    ]
}

print(json.dumps(va_report, indent=2))

## 9. Key Takeaways

| Concept | Takeaway |
|---------|----------|
| **CVSS scoring** | Always use v3.1 scores — v2 underestimates network-accessible vulns |
| **Prioritization** | CRITICAL first, then HIGH — never let score 9.8+ age past 24 hours |
| **CVE ≠ exploited** | Not every CVE has a public exploit — check CISA KEV catalog |
| **NVD API** | Free, machine-readable CVE data — build automation around it |
| **Service mapping** | Accurate version banners → accurate CVE matching |
| **Remediation SLAs** | Define SLAs before an incident — not during one |

### Useful Resources
- **NIST NVD:** https://nvd.nist.gov
- **CISA KEV (Known Exploited Vulnerabilities):** https://www.cisa.gov/known-exploited-vulnerabilities-catalog
- **Mitre CVE:** https://cve.mitre.org
- **Exploit-DB:** https://www.exploit-db.com

---
*Next: 11 — Incident Response & Digital Forensics*